# Stage 7: the unusual-activity flag (Isolation Forest)

This is **not a forecast**. It is a **smoke alarm**: it looks at *today* and asks "does this day look different from this stock's normal days?"

**The idea of Isolation Forest.** Imagine finding the odd one out in a crowd by asking random yes/no questions ("is your value above 3?", "is it below 0.7?"). A typical person takes many questions to single out, because many others look the same. A truly odd person is separated after only a few questions. Isolation Forest builds hundreds of random question trees and measures how quickly each day gets isolated. **Quick to isolate = unusual.**

**Four measures of "how unusual is today?"**, each compared with the stock's own previous 60 days:
- **Price move**: today's move as a multiple of the stock's normal wiggle
- **Day range**: the high-to-low swing, as a multiple of normal
- **Volume**: shares traded, as a multiple of normal
- **Opening gap**: the jump from yesterday's close to today's open, as a multiple of normal

This is **unsupervised**: nobody has labelled which days were "really" unusual, so we cannot compute an accuracy. We can only sanity-check it (does it light up on days everyone remembers?) and describe what it flags.

Kernel: **RiskWise (.venv)**.

In [ ]:
# Cell 1: tools and data
import sys
sys.path.append("../pipeline")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from sklearn.ensemble import IsolationForest
from anomaly import ANOMALY_COLS, to_model_input, main_reason
from features import make_features
from common import load_prices

cols = ["sym", "date"] + ANOMALY_COLS
tab = pd.read_parquet("../data/features.parquet", columns=cols).dropna()
tab["date"] = pd.to_datetime(tab["date"])
EXAM_START = tab["date"].max() - pd.DateOffset(years=3)
print(f"{len(tab):,} stock-days, {tab['sym'].nunique()} stocks")

In [ ]:
# Cell 2: a quiet day vs a famous day for Reliance
rel = tab[tab["sym"] == "RELIANCE"].set_index("date")
print("a typical day, and the Covid crash day (23 March 2020):")
rel.loc[["2019-06-12", "2020-03-23"], ANOMALY_COLS].round(2)

Read the table: a value of 1 means "exactly normal for this stock". On the crash day, the measures are several times normal.

## Step 1: teach the forest what "normal" looks like

We fit on a random sample of stock-days from **before the exam** only. We then choose the alarm level so that **about 1 in 100 stock-days** counts as unusual (about 15 stocks on an average day).

In [ ]:
# Cell 3: fit the Isolation Forest on pre-exam days only
before = tab[tab["date"] < EXAM_START - pd.Timedelta(days=10)]
sample = before.sample(400_000, random_state=0)
forest = IsolationForest(n_estimators=200, max_samples=1024, random_state=0, n_jobs=-1)
forest.fit(to_model_input(sample))

scores = forest.score_samples(to_model_input(sample))          # lower = more unusual
threshold = np.quantile(scores, 0.01)                          # flag the most unusual 1%
q99 = sample[ANOMALY_COLS].quantile(0.99).to_dict()             # used later to explain WHY a day is unusual
print("alarm level (scores below this are flagged):", round(float(threshold), 4))
print("each measure's 99th percentile:", {k: round(v, 1) for k, v in q99.items()})

In [ ]:
# Cell 4: flag every stock-day (training period AND exam period)
tab["score"] = forest.score_samples(to_model_input(tab))
tab["flag"] = tab["score"] < threshold
exam = tab[tab["date"] >= EXAM_START]
print("flag rate in the training period:", round(tab.loc[tab['date'] < EXAM_START, 'flag'].mean() * 100, 2), "%")
print("flag rate in the exam period (never seen by the forest):", round(exam["flag"].mean() * 100, 2), "%")

If the forest had memorised its training data, the flag rate on new days would drift far from 1%. Staying close means "unusual" keeps the same meaning over time.

## Sanity check 1: does it light up on days everyone remembers?

For each date, what share of all stocks got flagged? On an ordinary day it should be tiny; on a market-wide shock it should jump.

In [ ]:
# Cell 5: share of stocks flagged on famous days, against a typical day
daily_share = tab.groupby("date")["flag"].mean() * 100
events = {"2008-01-21": "Sensex falls about 7% in a day", "2008-10-24": "global financial crisis lows",
          "2020-03-13": "Covid panic", "2020-03-23": "Covid crash day", "2024-06-04": "general-election result day (inside the exam period)"}
rows = [{"date": d, "what happened": why, "% of stocks flagged": round(float(daily_share.get(pd.Timestamp(d), np.nan)), 1)} for d, why in events.items()]
rows.append({"date": "typical day", "what happened": "median over all days", "% of stocks flagged": round(float(daily_share.median()), 1)})
pd.DataFrame(rows).set_index("date")

## Sanity check 2: what do flagged days look like?

In [ ]:
# Cell 6: flagged days against normal days
summary = tab.groupby("flag")[ANOMALY_COLS].median().round(2)
summary.index = ["normal days", "flagged days"]
print("median 'multiple of normal' for each measure:")
display(summary)

## The reason behind each flag

A bare "unusual!" is not helpful. For every flagged day we find which measure is most extreme compared with its own 99th percentile and say so in plain English: "Volume 3.2x normal", "Price move 4.1x normal".

In [ ]:
# Cell 7: the reasons, and the most recent flagged days of five big stocks
flagged = tab[tab["flag"]].copy()
flagged["reason"] = flagged.apply(lambda r: main_reason(r, q99), axis=1)
flagged["kind"] = flagged["reason"].str.extract(r"^(Price move|Day range|Volume|Opening gap)")
print("what usually triggers a flag:")
print((flagged["kind"].value_counts(normalize=True) * 100).round(1).to_string())
BIG = ["RELIANCE", "TCS", "INFY", "HDFCBANK", "SBIN"]
print("\nmost recent flags on five big stocks:")
flagged[flagged["sym"].isin(BIG)].sort_values("date").tail(8)[["sym", "date", "reason"]].set_index("sym")

## Proof of no peeking

Today's measures compare today with the **previous** 60 days (we shifted the "normal" back by one day), so they can use no future information. The test below chops a stock's history right after one date, rebuilds the measures, and checks that day's four numbers did not change.

In [ ]:
# Cell 8: no-peeking check
df = load_prices("RELIANCE")
full = make_features(df)
d = full.dropna(subset=ANOMALY_COLS).index[-500]
chopped = make_features(df.loc[:d])
a, b = full.loc[d, ANOMALY_COLS].astype(float), chopped.loc[d, ANOMALY_COLS].astype(float)
print(pd.DataFrame({"full history": a, "chopped at d": b}).round(4).T.to_string())
assert np.allclose(a.values, b.values), "the measures changed, so they used the future!"
assert before["date"].max() < EXAM_START, "the forest saw exam days!"
print("\nIdentical, and the forest only saw days before the exam.")

In [ ]:
# Cell 9: save the forest for the nightly job
Path("../models").mkdir(exist_ok=True)
joblib.dump({"forest": forest, "threshold": float(threshold), "q99": q99}, "../models/isolation_forest.joblib")
print("saved ../models/isolation_forest.joblib")

## Reading the results honestly

- This flag says "today looks different from this stock's normal days". It does **not** say *why* in the real world (news, results, a big order), and it does not say the stock will fall or rise.
- About 1 in 100 days is flagged **by design**: we chose the alarm level. So "1% flagged" is not a result; the sanity checks (famous days, reasons) are.
- Thinly traded stocks can look "unusual" just because one large trade moved the volume. We should show the reason text so users can judge.

## Your turn

1. In Cell 3 change `0.01` to `0.05` (flag 5% of days). What changes in Cells 5 and 7?
2. Look at Cell 5: what share of stocks was flagged on the Covid crash day compared with a typical day?
3. Explain back: **why do we compare today with the previous 60 days rather than with the whole history?**